In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import (
    GridSearchCV,TimeSeriesSplit)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss)

import matplotlib.pyplot as plt
import seaborn as sns

#Load the dataset 
df = pd.read_csv("../data/atp_matches_2000_2019_clean.csv")

#Features 
X = [
    "rank_diff",
    "rank_points_diff",
    "age_diff",
    "height_diff",
    "different_hand",
    "elo_diff",
    "elo_diff_surface",
    "complete_diff",
    "serve_advantage_diff",
    "form_diff",
]

#target
y = 'result'

#Empty storage lists
dates = []
predictions = []
actuals = []
probabilities = []
best_params_list = []
cv_monitor = []

#Rolling setup
#Train on all previous matches, test on the next match
MIN_TRAIN = 2000 #min amount of matches to train on before testing
STEP=50 #model is updated after every 50 matches.

LOGIT = LogisticRegression(max_iter=5000, 
                           random_state=1)

pipe=Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LOGIT),
])

#Set the parameter grid for Logit
param_grid = {
    "clf__C": [0.05, 0.1, 0.2, 0.3, 1, 5, 10] #Note: for some reason LogisticRegression penalty term (c)
} #is determined 1/lambda, so smaller C -> greater regularization.

#Rolling Forecast Loop match-by-match
for i in range(MIN_TRAIN, len(df), STEP):
    train_data = df.iloc[:i]
    test_data = df.iloc[i:i+STEP]

    X_train = train_data[X]
    y_train = train_data[y]
    X_test  = test_data[X]
    y_test  = test_data[y]

    tscv = TimeSeriesSplit(n_splits=2)#tätä voisi kasvattaa jos kone jaksaa pyörittää

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=tscv,
        n_jobs=-1,
        scoring="neg_log_loss",
        refit=True,
        return_train_score=True
    )
    search.fit(X_train, y_train)

    # CV monitoring
    res = pd.DataFrame(search.cv_results_)
    best_row = res.loc[res["rank_test_score"].idxmin()]
    cv_monitor.append({
        "Index": i,
        "Date": test_data["tourney_date"].iloc[0],
        "Best_Params": search.best_params_,
        "Mean_Train_Score": float(best_row["mean_train_score"]),
        "Mean_Valid_Score": 
            float(best_row["mean_test_score"]),
    })

    #Store
    best_params_list.append(search.best_params_)
    dates.append(test_data["tourney_date"].iloc[0])
    preds = search.predict(X_test)
    probs = search.predict_proba(X_test)[:, 1]
    predictions.extend(preds)
    probabilities.extend(probs)
    actuals.extend(y_test.tolist())


In [ ]:

#Evaluation metrics
#Balanced Accuracy
accuracy = balanced_accuracy_score(actuals, predictions)
print(f"Balanced Accuracy: {accuracy:.3f}")

#ROC AUC
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(actuals, probabilities)
print(f"ROC AUC: {roc_auc:.3f}")

#Log Loss
logloss = log_loss(actuals, probabilities)
print(f"Log Loss: {logloss:.3f}")

#Brier score
from sklearn.metrics import brier_score_loss
brier_score = brier_score_loss(actuals, probabilities)
print(f"Brier Score: {brier_score:.3f}")


Balanced Accuracy: 0.677
ROC AUC: 0.745
Log Loss: 0.594
Brier Score: 0.205
